In [ ]:
import random
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS

random_state = 2
#need both bc ones for py gen and other controls np
np.random.seed(random_state)
random.seed(random_state)

path = "data/all_book_titles.txt"

#got code from other labs
with open(path, encoding="utf-8") as f:
    raw_lines = f.read().splitlines()

titles = [t.strip() for t in raw_lines if t.strip()]

#print("raw lines:", len(raw_lines))
#print("non-empty titles:", len(titles))
#should make one column table
df = pd.DataFrame({"title": titles})
df.head(10)

uni = df["title"].nunique()
#print("NEED TO BE REMOVED:", len(df) - uni)


#shows the stats of each title length
df["n_words"] = df["title"].str.split().str.len()
print(df["n_words"].describe())
#looked this up, splits titles into words, puts every word on its own row, counts them, output a table of 40 most common words
print(df["title"].str.lower().str.split().explode().value_counts().head(40))

#Found 142 duplicates want to remove them in the cleaning section. Titles have a median of 7 words (min 1, max 36). Noise words I plan to remove are the stop words, along with
#publishing words (edition, th, access), the "s" leftover from possesives, and other vague words (introduction, approach).


#cleaning
#drops dups
df = df.drop_duplicates(subset="title").reset_index(drop=True)
#print("Should be 0:", len(df) - uni)

noise = ["edition", "th", "s", "access"]
stop_list = list(ENGLISH_STOP_WORDS) + noise

#skips stop words and just keeps words that are in at least 5 titles -- maybe switch later i just think rarer words will show no pattern whatsoever
c_v = CountVectorizer(stop_words=stop_list, min_df=5)

#makes the vocab from c_v and then builds table with on row per title and on column per word, each row tells us how many times that word appears
X_counts = c_v.fit_transform(df["title"])

#basically same thing except each cell is weighted. lower score = appear more / higher score = rarer words
tfidf_v = TfidfVectorizer(stop_words=stop_list, min_df=5)
X_tfidf = tfidf_v.fit_transform(df["title"])

#print(len(c_v.get_feature_names_out()))

#LDA is given counts because it is a probability model. It treeats each title as if its words were drawn form topics, so it is built on how many times a word shows up.
#Raw counts fit perfectly with this model. While TF-IDF scores are scaleed dec that don't track word count. NMF only needs non-neg numbers, so we can use TF_IDF, which
#lowers the weight to words that appear more vis versa.

#Choose to remove dup words bc it would make words look more comman than they actually are inflating them
#using stop words bc they have nothing to do with topics
# added  "edition, th, access" to stop words bc it sjust publishing noise
#choose the min doc freq as 5 bc fewer is too rare to show any pattern at all





count    2373.000000
mean        7.142857
std         4.096361
min         1.000000
25%         4.000000
50%         6.000000
75%         9.000000
max        36.000000
Name: n_words, dtype: float64
title
and             932
of              576
the             489
edition)        321
to              320
a               304
in              304
for             258
introduction    236
with            180
an              175
edition         136
:               130
history         104
&               104
science         100
computer         88
approach         87
access           77
business         75
human            71
applications     68
principles       67
analysis         64
essentials       63
series           62
biology          61
art              58
theory           58
guide            56
fundamentals     56
statistics       53
card             53
chemistry        52
concepts         51
anatomy          51
design           50
data             50
world            49
systems         

In [9]:
from sklearn.decomposition import LatentDirichletAllocation, NMF

n_topics = 3

lda = LatentDirichletAllocation(n_components=n_topics, random_state=random_state)
lda.fit(X_counts)

nmf = NMF(n_components=n_topics, random_state=random_state)
nmf.fit(X_tfidf)

,"n_components n_components: int or {'auto'} or None, default='auto'Number of components. If `None`, all features are kept.If `n_components='auto'`, the number of components is automatically inferredfrom W or H shapes... versionchanged:: 1.4 Added `'auto'` value... versionchanged:: 1.6 Default value changed from `None` to `'auto'`.",3
,"random_state random_state: int, RandomState instance or None, default=NoneUsed for initialisation (when ``init`` == 'nndsvdar' or'random'), and in Coordinate Descent. Pass an int for reproducibleresults across multiple function calls.See :term:`Glossary <random_state>`.",2
,"init init: {'random', 'nndsvd', 'nndsvda', 'nndsvdar', 'custom'}, default=NoneMethod used to initialize the procedure.Valid options:- `None`: 'nndsvda' if n_components <= min(n_samples, n_features), otherwise random.- `'random'`: non-negative random matrices, scaled with: `sqrt(X.mean() / n_components)`- `'nndsvd'`: Nonnegative Double Singular Value Decomposition (NNDSVD) initialization (better for sparseness)- `'nndsvda'`: NNDSVD with zeros filled with the average of X (better when sparsity is not desired)- `'nndsvdar'` NNDSVD with zeros filled with small random values (generally faster, less accurate alternative to NNDSVDa for when sparsity is not desired)- `'custom'`: Use custom matrices `W` and `H` which must both be provided... versionchanged:: 1.1 When `init=None` and n_components is less than n_samples and n_features defaults to `nndsvda` instead of `nndsvd`.",None
,"solver solver: {'cd', 'mu'}, default='cd'Numerical solver to use:- 'cd' is a Coordinate Descent solver.- 'mu' is a Multiplicative Update solver... versionadded:: 0.17 Coordinate Descent solver... versionadded:: 0.19 Multiplicative Update solver.",'cd'
,"beta_loss beta_loss: float or {'frobenius', 'kullback-leibler', 'itakura-saito'}, default='frobenius'Beta divergence to be minimized, measuring the distance between Xand the dot product WH. Note that values different from 'frobenius'(or 2) and 'kullback-leibler' (or 1) lead to significantly slowerfits. Note that for beta_loss <= 0 (or 'itakura-saito'), the inputmatrix X cannot contain zeros. Used only in 'mu' solver... versionadded:: 0.19",'frobenius'
,"tol tol: float, default=1e-4Tolerance of the stopping condition.",0.0001
,"max_iter max_iter: int, default=200Maximum number of iterations before timing out.",200
,"alpha_W alpha_W: float, default=0.0Constant that multiplies the regularization terms of `W`. Set it to zero(default) to have no regularization on `W`... versionadded:: 1.0",0.0
,"alpha_H alpha_H: float or ""same"", default=""same""Constant that multiplies the regularization terms of `H`. Set it to zero tohave no regularization on `H`. If ""same"" (default), it takes the same value as`alpha_W`... versionadded:: 1.0",'same'
,"l1_ratio l1_ratio: float, default=0.0The regularization mixing parameter, with 0 <= l1_ratio <= 1.For l1_ratio = 0 the penalty is an elementwise L2 penalty(aka Frobenius Norm).For l1_ratio = 1 it is an elementwise L1 penalty.For 0 < l1_ratio < 1, the penalty is a combination of L1 and L2... versionadded:: 0.17 Regularization parameter *l1_ratio* used in the Coordinate Descent solver.",0.0
,"verbose verbose: int, default=0Whether to be verbose.",0
